# `run_idmatcherai_by_trial` — Google Colab edition

Groups `session_*` folders by trial number (`_N<n>_` in the folder name),
validates compatibility within each group, runs **idmatcherai** to compute
identity correspondences across video segments, and writes a combined CSV +
JSON report to Google Drive.

**New in the Colab version:** `DROP_PHARMA_GROUPS` switch that removes all
trials whose `Group` (from `trials summary GOT.xlsx`) contains *naloxone* or
*morphine* before any matching is done.

Run cells top-to-bottom. Configurable knobs live in the **Configuration**
cell. See the final section *How to Run This Notebook* for the full
checklist.

## GPU Runtime Setup

`idmatcherai` uses a PyTorch convolutional network to compare identification
images across video segments — it **does** benefit from a GPU. The
notebook's own bookkeeping (folder walking, JSON/CSV parsing, Excel reading)
is CPU-bound, but those are seconds; the bulk of wall time is inside
`idmatcherai`, which runs CUDA-accelerated when a GPU is present.

**Enable the T4 GPU now:**

> **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

The next cell verifies that the GPU is visible.

In [ ]:
import subprocess, sys

print("── nvidia-smi ──────────────────────────────────────────────")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print(f"[WARN] nvidia-smi unavailable: {e}")
    print("       The pipeline will still run on CPU but will be slower.")

print("── PyTorch CUDA check ──────────────────────────────────────")
try:
    import torch
    print(f"torch version       : {torch.__version__}")
    print(f"CUDA available      : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA device         : {torch.cuda.get_device_name(0)}")
    else:
        print("[WARN] CUDA not available. idmatcherai will fall back to CPU.")
        print("       If you intended GPU: Runtime → Change runtime type → T4 GPU.")
except ImportError:
    print("[INFO] torch not installed yet — it will be installed with idtrackerai below.")

## Google Drive Paths and File Discovery

The notebook reads every input and writes every output under
`/content/drive/MyDrive/`. The expected Drive layout (you can override
`PROJECT_DIR` below):

```
/content/drive/MyDrive/GOT R SCRIPTS/
├── checked_sessions_choice_exp/       ← your session_* folders (already on Drive)
│   ├── session_..._N1_segment1/
│   ├── session_..._N1_segment2/
│   └── ...
├── datasets/
│   └── trials summary GOT.xlsx        ← needed when DROP_PHARMA_GROUPS = True
└── idmatcher_results/                 ← created automatically; outputs land here
    └── <timestamp>/
        ├── idmatcher_identity_correspondences.csv
        ├── run_report.json
        └── logs/
```

> The folder named `checked_sessions_choice_exp/` is the *only* one assumed
> to already exist on Drive — upload everything else from the delivery
> folder.

### Step 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Step 2 — Configuration

Edit these constants if your Drive layout differs. All paths are
`pathlib.Path` objects — they are inert strings until the validation cell
actually touches Drive.

In [ ]:
from pathlib import Path

# ── Where everything lives on Drive ──────────────────────────────────────────
# Shared-folder access
# • Google Workspace Shared Drive: the path below is correct as-is.
# • Personal Google account: right-click the folder in "Shared with me"
#   → "Organise" → "Add shortcut to Drive" (place in My Drive),
#   then change "Shareddrives" to "MyDrive" in the line below.
DRIVE_ROOT          = Path("/content/drive/Shareddrives")
PROJECT_DIR         = DRIVE_ROOT / "idmatcherai"
SESSIONS_ROOT       = PROJECT_DIR / "sessions"
TRIALS_SUMMARY_XLSX = PROJECT_DIR / "datasets" / "trials summary GOT.xlsx"
OUTPUT_DIR          = SESSIONS_ROOT / "idmatcher_results"

# ── Pharma filter ────────────────────────────────────────────────────────────
# True  → drop every trial whose `Group` in trials summary GOT.xlsx contains
#         'naloxone', 'morphine', or the known typo 'moprhine' (case-insensitive).
# False → keep all trials and skip reading the Excel.
DROP_PHARMA_GROUPS  = True

# ── Run mode ─────────────────────────────────────────────────────────────────
# DRY_RUN = True  → validate everything and print idmatcherai commands without
#                   executing them. Useful for a first end-to-end sanity check.
# ONLY_TRIALS     → empty list = process all trials. Otherwise restrict to e.g.
#                   [5, 7] to test on a couple of non-pharma trials first.
DRY_RUN     = False
ONLY_TRIALS = []
TP1_SWITCH  = True   # True -> exclude every Segment1 session from matching


### Step 3 — Install dependencies

Installs `idtrackerai` (which provides the `idmatcherai` CLI and pulls in
PyTorch + OpenCV), plus pandas / openpyxl for the pharma filter. Expect a
few minutes on the first run; subsequent re-runs are fast because Colab
caches wheels.

In [ ]:
%pip install -q idtrackerai pandas openpyxl
print("Done.")

### Step 4 — Validate paths

Confirms every required input exists *now*, creates the output root, and
prints a readable summary. If anything is wrong this is where you'll see a
clear, actionable error.

In [ ]:
def validate_paths(*, drop_pharma: bool):
    """Check Drive paths and print a summary. Raises FileNotFoundError on any miss."""
    print("── Path validation ─────────────────────────────────────────")
    print(f"PROJECT_DIR         : {PROJECT_DIR}")
    print(f"SESSIONS_ROOT       : {SESSIONS_ROOT}")
    print(f"TRIALS_SUMMARY_XLSX : {TRIALS_SUMMARY_XLSX}")
    print(f"OUTPUT_DIR          : {OUTPUT_DIR}")
    print()

    if not DRIVE_ROOT.exists():
        raise FileNotFoundError(
            f"Drive is not mounted at {DRIVE_ROOT}. "
            "Re-run the 'Mount Drive' cell above."
        )

    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f"PROJECT_DIR not found: {PROJECT_DIR}\n"
            "Edit the configuration cell so PROJECT_DIR points at your project folder on Drive."
        )

    if not SESSIONS_ROOT.exists():
        raise FileNotFoundError(
            f"SESSIONS_ROOT not found: {SESSIONS_ROOT}\n"
            "Make sure the checked_sessions_choice_exp/ folder is present on Drive."
        )

    sess = sorted(p for p in SESSIONS_ROOT.iterdir()
                  if p.is_dir() and p.name.lower().startswith("session_"))
    if not sess:
        raise FileNotFoundError(
            f"SESSIONS_ROOT exists but contains no session_* folders: {SESSIONS_ROOT}"
        )
    print(f"Found {len(sess)} session_* folder(s) under SESSIONS_ROOT.")

    if drop_pharma:
        if not TRIALS_SUMMARY_XLSX.exists():
            raise FileNotFoundError(
                f"DROP_PHARMA_GROUPS=True but the trials summary is missing:\n  {TRIALS_SUMMARY_XLSX}\n"
                "Either upload it to that path or set DROP_PHARMA_GROUPS = False."
            )
        print(f"Trials summary OK   : {TRIALS_SUMMARY_XLSX.name}")
    else:
        print("DROP_PHARMA_GROUPS=False — trials summary not required.")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Output root ready   : {OUTPUT_DIR}")
    print("── Validation OK ───────────────────────────────────────────")

validate_paths(drop_pharma=DROP_PHARMA_GROUPS)


### Validate a Specific Session Group

Use the `validate_group` function to check the internal consistency and compatibility of a custom list of session folders. Modify the `sess_list` variable in the cell below with the paths to the session folders you want to validate.

## Pipeline functions

All helpers from the original `run_idmatcherai_by_trial.py` are preserved
verbatim, with two changes:

1. `build_idmatcherai_cmd()` now uses `shutil.which("idmatcherai")` first
   (with a `python -m idtrackerai.extra_tools.idmatcherai` fallback) —
   no more hard-coded conda paths.
2. `main()` is renamed `run_matching_pipeline(...)` and accepts the
   notebook config variables as arguments instead of `argparse`.

A new `drop_pharma_trial_groups()` function reads the Excel and filters the
`groups` dict before any matching happens.

In [ ]:
import csv, json, os, re, shutil, subprocess, sys, tempfile
from datetime import datetime

# ── Regexes (identical to the original script) ───────────────────────────────
_TRIAL_RE   = re.compile(r"trial(\d+)", re.IGNORECASE)
_SEGMENT_RE = re.compile(r"segment_?(\d+)",  re.IGNORECASE)

def parse_trial(name: str):
    m = _TRIAL_RE.search(name)
    return int(m.group(1)) if m else None

def parse_segment(name: str):
    m = _SEGMENT_RE.search(name)
    return int(m.group(1)) if m else None

REQUIRED_SUBFOLDERS  = ["accumulation", "identification_images", "trajectories"]
REQUIRED_DATA_POLICY = "idmatcher.ai"

def load_session_json(session_dir: Path):
    """Return (dict, error_str). Handles Infinity / NaN in older JSON."""
    p = session_dir / "session.json"
    if not p.exists():
        return None, "session.json not found"
    try:
        text = p.read_text(encoding="utf-8", errors="ignore")
        text = text.replace(": Infinity", ": 1e308").replace(":Infinity", ":1e308")
        text = text.replace(": NaN",      ": null" ).replace(":NaN",     ":null" )
        return json.loads(text), None
    except Exception as exc:
        return None, f"session.json parse error: {exc}"

def check_log(session_dir: Path):
    for fname in ("idtrackerai.txt", "idtrackerai.log"):
        p = session_dir / fname
        if p.exists():
            lines = p.read_text(encoding="utf-8", errors="ignore").splitlines()
            err = [
                l.strip() for l in lines
                if (
                    l.strip().upper().startswith("ERROR")
                    or " ERROR " in l.upper()
                    or "CRITICAL" in l.upper()
                    or "Traceback" in l
                )
                and "ignoring the above errors" not in l.lower()
            ]
            return fname, err[:5]
    return None, []

def validate_session(session_dir: Path):
    hard_fails, warnings = [], []
    sj, err = load_session_json(session_dir)
    if err:
        hard_fails.append(err)
        return {}, hard_fails, warnings

    missing = [f for f in REQUIRED_SUBFOLDERS if not (session_dir / f).exists()]
    if missing:
        hard_fails.append(f"missing required subfolder(s): {', '.join(missing)}")

    dp = sj.get("data_policy", "")
    if dp != REQUIRED_DATA_POLICY:
        hard_fails.append(
            f"data_policy='{dp}' — must be '{REQUIRED_DATA_POLICY}'. "
            "Re-track the video with: Settings > Data policy > idmatcher.ai"
        )

    log_fname, log_errors = check_log(session_dir)
    if log_errors:
        warnings.append(f"{log_fname} contains possible error lines: " + " | ".join(log_errors))

    meta = {
        "id_image_size": sj.get("id_image_size"),
        "n_animals":     sj.get("n_animals") or sj.get("number_of_animals"),
        "version":       sj.get("version"),
        "data_policy":   dp,
    }
    return meta, hard_fails, warnings

def validate_group(sess_list):
    all_meta, group_hard, group_warn = {}, [], []
    for s in sess_list:
        meta, hf, wn = validate_session(s)
        all_meta[s.name] = meta
        for msg in hf: group_hard.append(f"{s.name}: {msg}")
        for msg in wn: group_warn.append(f"{s.name}: {msg}")

    # Model-completeness check: block sessions whose training was never finished
    # (accumulation/ contains only 0-byte tmp_* files — no finalised checkpoint).
    def _model_is_complete(session_dir: Path):
        """Return (ok: bool, reason: str)."""
        acc = session_dir / "accumulation"
        if not acc.exists():
            return False, "accumulation/ missing"
        try:
            final = [f for f in acc.iterdir()
                     if f.suffix == ".pt" and "tmp_" not in f.name and f.stat().st_size > 0]
        except PermissionError:
            return True, ""  # can't check — assume ok
        if not final:
            return False, (
                "no finalised model checkpoint in accumulation/ "
                "(only 0-byte or tmp_* files found — session was never tracked to completion)"
            )
        return True, ""

    for s in sess_list:
        ok, reason = _model_is_complete(s)
        if not ok:
            group_hard.append(f"{s.name}: {reason}")

    sizes = {
        tuple(m["id_image_size"]) if isinstance(m.get("id_image_size", []), list)
        else m.get("id_image_size")
        for m in all_meta.values() if m.get("id_image_size") is not None
    }
    if len(sizes) > 1:
        detail = ", ".join(f"{n}={m['id_image_size']}" for n, m in all_meta.items())
        group_hard.append(
            f"id_image_size mismatch across sessions ({detail}). "
            "All segments of a trial must be tracked with the same image size."
        )

    # n_animals must be identical (was a warning; now a hard fail).
    n_set = {m["n_animals"] for m in all_meta.values() if m.get("n_animals")}
    if len(n_set) > 1:
        detail = ", ".join(f"{n}={m['n_animals']}" for n, m in all_meta.items())
        group_hard.append(
            f"n_animals differs across sessions ({detail}). "
            "idmatcher.ai cannot reliably match identities when groups have different "
            "animal counts. Re-track the outlier session(s) with the correct number "
            "of animals, or exclude the trial via ONLY_TRIALS."
        )

    # Model architecture must be identical across the group.
    # Read 'model' from accumulation/model_params.json (IdCNN vs ResNet18).
    # Missing/unreadable JSON returns None and is treated as 'unknown', so a
    # truly empty group_arch_set won't trigger a false mismatch.
    def _read_model_arch(session_dir):
        mp = session_dir / "accumulation" / "model_params.json"
        if not mp.exists():
            return None
        try:
            return json.loads(mp.read_text(encoding="utf-8")).get("model")
        except Exception:
            return None

    arch_map = {s.name: _read_model_arch(s) for s in sess_list}
    arch_set = {a for a in arch_map.values() if a is not None}
    if len(arch_set) > 1:
        detail = ", ".join(f"{n}={a}" for n, a in arch_map.items())
        group_hard.append(
            f"identification model architecture differs across sessions ({detail}). "
            "Mixing IdCNN and ResNet18 in the same group will produce unreliable "
            "matches. Re-track the outlier segment(s) with the same model, or "
            "exclude the trial via ONLY_TRIALS."
        )

    v_set = {m["version"] for m in all_meta.values() if m.get("version")}
    if len(v_set) > 1:
        detail = ", ".join(f"{n}={m['version']}" for n, m in all_meta.items())
        group_warn.append(
            f"idtracker.ai version differs across sessions ({detail}) — "
            "this can cause poor matchings"
        )

    return all_meta, group_hard, group_warn

def build_idmatcherai_cmd():
    """Locate the idmatcherai entry point. Colab-friendly: no hard-coded paths."""
    exe = shutil.which("idmatcherai")
    if exe:
        return [exe]
    # Fallback: run as a module under the current Python.
    try:
        import importlib
        importlib.import_module("idtrackerai.extra_tools.idmatcherai")
        return [sys.executable, "-m", "idtrackerai.extra_tools.idmatcherai"]
    except Exception:
        raise FileNotFoundError(
            "Cannot find idmatcherai. Did the 'Install dependencies' cell finish?\n"
            "Try re-running:  %pip install -q idtrackerai"
        )

def choose_master(sess_list):
    keyed = [(parse_segment(p.name) or 9999, p) for p in sess_list]
    return sorted(keyed, key=lambda t: t[0])[0][1]

def read_assignments_csv(csv_path: Path) -> dict:
    result = {}
    if not csv_path.exists():
        return result
    try:
        with csv_path.open(encoding="utf-8") as f:
            reader = csv.DictReader(f, skipinitialspace=True)
            for row in reader:
                try:
                    identity   = int(row["identity"].strip())
                    assignment = int(row["assignment"].strip())
                    direct     = float(row["direct score"].strip())
                    indirect   = float(row["indirect score"].strip())
                    result[assignment] = {
                        "matching_id":    identity,
                        "direct_score":   round(direct,   4),
                        "indirect_score": round(indirect, 4),
                    }
                except (KeyError, ValueError):
                    continue
    except Exception as exc:
        print(f"  [WARN] Could not parse {csv_path}: {exc}")
    return result


def log_indicates_crash(log_path: Path) -> tuple:
    """Return (crashed: bool, excerpt: str). Detects exit-0 crashes in idtrackerai logs."""
    text = log_path.read_text(encoding="utf-8", errors="ignore")
    for needle in ("RuntimeError", "Traceback (most recent call last)",
                   "CRITICAL", "KeyError", "ValueError"):
        if needle in text:
            for line in text.splitlines():
                if needle in line:
                    return True, line.strip()[:200]
    return False, ""


### Pharma group filter

Reads `trials summary GOT.xlsx` (columns `Trial` and `Group`), finds every
trial whose `Group` contains `naloxone`, `morphine`, or the typo `moprhine`
(all case-insensitive, NaN-safe, whitespace-tolerant), and drops those
trial IDs *as complete groups* from the `groups` dict.

In [ ]:
import pandas as pd

PHARMA_NEEDLES = ("naloxone", "morphine", "moprhine")  # third entry handles the known typo

def drop_pharma_trial_groups(groups: dict, trials_xlsx: Path) -> dict:
    """
    Drop every (trial → session list) entry whose Trial appears in the Excel
    with a Group containing any pharma needle.

    Returns the filtered dict and logs what was removed.
    """
    if not trials_xlsx.exists():
        raise FileNotFoundError(
            f"Trials summary not found: {trials_xlsx}\n"
            "Either upload it to that path or set DROP_PHARMA_GROUPS = False."
        )

    df = pd.read_excel(trials_xlsx)
    missing_cols = [c for c in ("Trial", "Group") if c not in df.columns]
    if missing_cols:
        raise KeyError(
            f"trials summary GOT.xlsx is missing required column(s): {missing_cols}\n"
            f"Found columns: {list(df.columns)}"
        )

    g_norm = df["Group"].fillna("").astype(str).str.strip().str.lower()
    is_pharma = g_norm.apply(lambda s: any(n in s for n in PHARMA_NEEDLES))

    # Coerce Trial to int where possible; drop any rows whose Trial can't be parsed.
    trials_int = pd.to_numeric(df["Trial"], errors="coerce").astype("Int64")
    bad_mask = is_pharma & trials_int.notna()
    bad_trials = set(int(t) for t in trials_int[bad_mask].dropna().unique())

    print("── Pharma filter ──────────────────────────────────────────")
    print(f"Excel rows total            : {len(df)}")
    print(f"Excel rows matching pharma  : {int(bad_mask.sum())}")
    print(f"Distinct pharma trial IDs   : {len(bad_trials)} -> {sorted(bad_trials)}")

    if bad_trials:
        # Per-trial breakdown of which group(s) caused the drop.
        for t in sorted(bad_trials):
            groups_for_t = df.loc[trials_int == t, "Group"].dropna().unique().tolist()
            print(f"   Trial {t:>2}: groups = {groups_for_t}")

    present_pharma = sorted(bad_trials & set(groups))
    print(f"Pharma trials present in session folders: {len(present_pharma)} -> {present_pharma}")

    kept = {t: s for t, s in groups.items() if t not in bad_trials}
    removed_sessions = sum(len(s) for t, s in groups.items() if t in bad_trials)
    print(f"Sessions removed from this run          : {removed_sessions}")
    print(f"Trials remaining for matching           : {len(kept)} -> {sorted(kept)}")
    print("───────────────────────────────────────────────────────────")
    return kept


### Main pipeline

Equivalent to `main()` in the original script. Accepts the notebook
configuration as arguments and inserts the pharma filter right after the
groups are built.

In [ ]:
def run_matching_pipeline(*, sessions_root: Path, output_dir: Path,
                          trials_xlsx: Path, drop_pharma: bool,
                          dry_run: bool, only_trials, tp1_switch: bool = False):
    only = set(only_trials) if only_trials else set()

    stamp    = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_root = output_dir / stamp
    logs_dir = out_root / "logs"
    logs_dir.mkdir(parents=True, exist_ok=True)
    assignments_dir = out_root / "assignments"
    matrices_dir    = out_root / "matrices"
    plots_dir       = out_root / "plots"
    assignments_dir.mkdir(parents=True, exist_ok=True)
    matrices_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    idm_base = build_idmatcherai_cmd()
    # Fix A (corrected): use a wrapper script that patches torch.load before importing
    # idtrackerai. PYTHONSTARTUP is only honoured in interactive Python (REPL), not when
    # running a script/module via subprocess — so the previous PYTHONSTARTUP approach
    # silently did nothing. The wrapper is passed as the Python program to execute.
    _wrapper = Path(tempfile.mkdtemp()) / "_idmatcherai_wrapper.py"
    # The wrapper:
    #  1. Patches torch.load so MPS-tagged macOS checkpoints load on Linux/CUDA.
    #  2. Re-executes the installed idmatcherai console script via runpy. The
    #     console script knows the correct entry-function name (setuptools-generated);
    #     we do not.
    _wrapper.write_text(
        "\n".join([
            "import torch as _t",
            "_orig = _t.load",
            "def _safe(*a, **kw):",
            "    kw.setdefault('map_location', 'cuda' if _t.cuda.is_available() else 'cpu')",
            "    return _orig(*a, **kw)",
            "_t.load = _safe",
            "import runpy, shutil, sys",
            "_exe = shutil.which('idmatcherai')",
            "if _exe is None:",
            "    raise RuntimeError('idmatcherai console script not found on PATH')",
            "sys.argv[0] = _exe",
            "runpy.run_path(_exe, run_name='__main__')",
        ]) + "\n",
        encoding="utf-8",
    )

    # Pre-flight smoke test: run the wrapper with --help. If the installed
    # idtrackerai is missing the console script or has an import error, fail
    # fast with a clear message rather than crashing every trial in the loop.
    _smoke = subprocess.run(
        [sys.executable, str(_wrapper), "--help"],
        capture_output=True, text=True,
    )
    _smoke_out = (_smoke.stdout or "") + (_smoke.stderr or "")
    if _smoke.returncode != 0 and ("ImportError" in _smoke_out
                                   or "ModuleNotFoundError" in _smoke_out
                                   or "console script not found" in _smoke_out):
        raise RuntimeError(
            "idmatcherai wrapper smoke-test failed. Wrapper output: " + _smoke_out
        )

    print(f"idmatcherai : {' '.join(idm_base)}")
    print(f"Sessions    : {sessions_root}")
    print(f"Output      : {out_root}")
    print(f"Mode        : {'DRY RUN' if dry_run else 'EXECUTE'}\n")

    # ── Collect and group sessions ────────────────────────────────────────────
    all_dirs = sorted(
        [p for p in sessions_root.iterdir()
         if p.is_dir() and p.name.lower().startswith("session_")],
        key=lambda p: p.name.lower()
    )

    groups, unparsed = {}, []
    for d in all_dirs:
        trial = parse_trial(d.name)
        if trial is None or not (1 <= trial <= 99):
            unparsed.append(d.name)
            continue
        if only and trial not in only:
            continue
        if tp1_switch and parse_segment(d.name) == 1:
            continue
        groups.setdefault(trial, []).append(d)

    if unparsed:
        print(f"[NOTE] {len(unparsed)} folder(s) skipped (could not parse trial number):")
        for u in unparsed:
            print(f"       {u}")
        print()

    if not groups:
        raise RuntimeError("No valid session folders found in " + str(sessions_root))

    # ── Pharma filter (new in Colab edition) ──────────────────────────────────
    if drop_pharma:
        groups = drop_pharma_trial_groups(groups, trials_xlsx)
        if not groups:
            raise RuntimeError(
                "All trials were removed by the pharma filter. "
                "Set DROP_PHARMA_GROUPS = False to keep them."
            )

    # ── Per-trial processing ──────────────────────────────────────────────────
    report, all_rows, all_segs_seen = [], [], set()

    for trial, sess_list in sorted(groups.items()):
        sess_list = sorted(sess_list,
                           key=lambda p: (parse_segment(p.name) or 999, p.name))

        sep = "=" * 60
        print(f"\n{sep}")
        print(f" Trial N{trial}  ({len(sess_list)} segment(s) found)")
        for s in sess_list:
            print(f"   seg{parse_segment(s.name) or '?'}: {s.name}")

        if len(sess_list) < 2:
            msg = "only 1 session found -- need >= 2 to match"
            print(f"[SKIP] {msg}")
            report.append({"trial": trial, "status": "skipped", "reason": msg})
            continue

        all_meta, hard_fails, warnings = validate_group(sess_list)
        for w in warnings: print(f"[WARN] {w}")
        for f in hard_fails: print(f"[FAIL] {f}")
        if hard_fails:
            report.append({
                "trial": trial, "status": "failed_precheck",
                "hard_fails": hard_fails, "warnings": warnings,
            })
            continue

        master   = choose_master(sess_list)
        matching = [s for s in sess_list if s != master]
        cmd      = [sys.executable, str(_wrapper)] + [str(master)] + [str(s) for s in matching]
        log_path = logs_dir / f"trial{trial:02d}_master_{master.name}.log"

        print(f"\n[CMD] master : {master.name}")
        for m in matching: print(f"[CMD]   match: {m.name}")
        print(f"[CMD] {' '.join(cmd)}")

        if dry_run:
            report.append({"trial": trial, "status": "dry_run",
                           "cmd": cmd, "warnings": warnings})
            continue

        print(f"[RUN] Writing log to {log_path.name} …")
        with log_path.open("w", encoding="utf-8") as lf:
            proc = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT,
                                  cwd=str(sessions_root))

        _crashed, _crash_line = log_indicates_crash(log_path)
        if proc.returncode != 0 or _crashed:
            _excerpt = _crash_line or f"non-zero exit code {proc.returncode}"
            print(f"[FAIL] idmatcherai crashed: {_excerpt}")
            print(f"[FAIL] See log: {log_path}")
            report.append({
                "trial": trial, "status": "idmatcher_crashed",
                "exit_code": proc.returncode, "crash_excerpt": _excerpt,
                "log": str(log_path),
                "master": master.name, "matching": [s.name for s in matching],
                "warnings": warnings,
            })
            continue

        print("[OK]  idmatcherai finished")

        master_seg = parse_segment(master.name) or 0
        n_fish     = (all_meta.get(master.name) or {}).get("n_animals") or 0

        seg_id_map, seg_score_map = {}, {}
        for fid in range(1, n_fish + 1):
            seg_id_map.setdefault(fid, {})[master_seg] = fid
        all_segs_seen.add(master_seg)

        missing_csvs = []
        for m in matching:
            m_seg    = parse_segment(m.name) or 0
            csv_path = m / "matching_results" / master.name / "assignments.csv"
            assign   = read_assignments_csv(csv_path)

            if not assign:
                missing_csvs.append(str(csv_path))
                print(f"  [WARN] assignments.csv not found or empty: {csv_path}")
                for fid in range(1, n_fish + 1):
                    seg_id_map.setdefault(fid, {})[m_seg] = None
                all_segs_seen.add(m_seg)
                continue

            for master_fid, v in assign.items():
                seg_id_map.setdefault(master_fid, {})[m_seg] = v["matching_id"]
                seg_score_map[(master_fid, m_seg)] = (v["direct_score"], v["indirect_score"])
            all_segs_seen.add(m_seg)
            # copy idmatcherai output files to organised subdirs
            _results_dir = m / "matching_results" / master.name
            _stem = f"trial{trial:02d}_seg{m_seg}_vs_seg{master_seg}"
            if _results_dir.exists():
                _all = sorted(_results_dir.rglob("*"))
                _visible = [str(_f.relative_to(_results_dir)) for _f in _all if _f.is_file() and not _f.name.startswith("._")]
                print(f"  [FILES] {_results_dir.name}: {_visible if _visible else chr(40)+chr(101)+chr(109)+chr(112)+chr(116)+chr(121)+chr(41)}")
                for _f in _all:
                    if not _f.is_file():
                        continue
                    if _f.name.startswith("._"):
                        continue
                    if _f.suffix == ".png":
                        shutil.copy2(_f, plots_dir / f"{_stem}_{_f.name.replace('_matches', '_matching_matrix')}")
                    elif _f.name == "assignments.csv":
                        shutil.copy2(_f, assignments_dir / f"{_stem}_{_f.name}")
                    elif _f.suffix == ".csv":
                        shutil.copy2(_f, matrices_dir / f"{_stem}_{_f.name.replace('_matches', '_matching_matrix')}")

        # Fix D: omit rows when no non-master assignments were produced.
        _segs_matched = [parse_segment(m.name) or 0 for m in matching]
        _any_assigned = any(
            seg_id_map.get(fid, {}).get(seg) is not None
            for seg in _segs_matched
            for fid in (range(1, n_fish + 1) if n_fish > 0 else [])
        )
        if not _any_assigned:
            print(f"[SKIP] No assignments written for trial {trial} -- "
                  "rows omitted from CSV (trial still appears in run_report.json).")
            report.append({
                "trial": trial, "status": "idmatcher_no_output",
                "master": master.name, "matching": [s.name for s in matching],
                "n_fish": n_fish, "warnings": warnings,
                "missing_csvs": missing_csvs, "log": str(log_path),
            })
            continue

        fish_ids = list(range(1, n_fish + 1)) if n_fish > 0 else sorted(
            set().union(*[d.keys() for d in seg_id_map.values()])
        )
        for fid in fish_ids:
            row = {"trial": trial, "fish_id_master": fid,
                   "master_segment": master_seg, "master_session": master.name}
            for seg in sorted(seg_id_map.get(fid, {}).keys()):
                row[f"seg{seg}_fish_id"] = seg_id_map[fid].get(seg, "")
                sc = seg_score_map.get((fid, seg))
                if sc is not None:
                    row[f"seg{seg}_direct_score"]    = sc[0]
                    row[f"seg{seg}_indirect_score"]  = sc[1]
            all_rows.append(row)

        report.append({
            "trial": trial, "status": "ok",
            "master": master.name, "master_seg": master_seg,
            "matching": [s.name for s in matching], "n_fish": n_fish,
            "warnings": warnings, "missing_csvs": missing_csvs,
            "log": str(log_path),
        })

    # ── Build final CSV ──────────────────────────────────────────────────────
    seg_nums   = sorted(all_segs_seen)
    fixed_cols = ["trial", "fish_id_master", "master_segment", "master_session"]
    score_cols = []
    for seg in seg_nums:
        score_cols += [f"seg{seg}_fish_id", f"seg{seg}_direct_score", f"seg{seg}_indirect_score"]
    fieldnames = fixed_cols + score_cols

    csv_out = out_root / "idmatcher_identity_correspondences.csv"
    with csv_out.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in sorted(all_rows, key=lambda r: (r["trial"], r["fish_id_master"])):
            writer.writerow(row)

    report_out = out_root / "run_report.json"
    report_out.write_text(json.dumps(report, indent=2), encoding="utf-8")

    print(f"\n{'=' * 60}")
    n_ok   = sum(1 for r in report if r["status"] == "ok")
    n_fail = sum(1 for r in report if r["status"] in ("failed_precheck", "idmatcher_crashed"))
    n_skip = sum(1 for r in report if r["status"] in ("skipped", "dry_run"))
    print(f"Trials processed : {n_ok} OK | {n_fail} failed | {n_skip} skipped")
    print(f"Fish rows written: {len(all_rows)}")
    print(f"Output CSV       : {csv_out}")
    print(f"Run report       : {report_out}")
    if dry_run:
        print("\n[DRY RUN] No idmatcherai commands were executed.")
    return out_root

## Run the pipeline

In [ ]:
out_root = run_matching_pipeline(
    sessions_root=SESSIONS_ROOT,
    output_dir   =OUTPUT_DIR,
    trials_xlsx  =TRIALS_SUMMARY_XLSX,
    drop_pharma  =DROP_PHARMA_GROUPS,
    dry_run      =DRY_RUN,
    only_trials  =ONLY_TRIALS,
    tp1_switch   =TP1_SWITCH,
)
print(f"\nAll outputs are under: {out_root}")

## How to Run This Notebook

1. **Upload the project folder to Google Drive.** Put `run_idmatcherai_by_trial.ipynb`,
   `datasets/trials summary GOT.xlsx`, and your existing
   `checked_sessions_choice_exp/` folder under one parent on Drive — by
   default `My Drive/GOT R SCRIPTS/`. The exact layout to mirror is shown
   in *Google Drive Paths and File Discovery* above.
2. **Open the notebook** in Colab (right-click the `.ipynb` in Drive →
   *Open with* → *Google Colaboratory*).
3. **Enable T4 GPU**: *Runtime → Change runtime type → Hardware accelerator
   → T4 GPU → Save*. Then *Runtime → Run all* once to install deps, or run
   cells one by one.
4. **Run the setup cells** (GPU check, Mount Drive, Configuration, Install
   dependencies, Validate paths). The validation cell prints a summary —
   read it before moving on.
5. **Confirm the Drive paths** in the Configuration cell match where you
   put your data. The defaults assume `My Drive/GOT R SCRIPTS/…`.
6. **Set the switches** in the Configuration cell:
   - `DROP_PHARMA_GROUPS = True` to skip naloxone/morphine trials, `False` to keep them.
   - `DRY_RUN = True` for a no-execute trial; `False` for the real run.
   - `ONLY_TRIALS = [5, 7]` to restrict to specific trial numbers; `[]` for all.
7. **Run the pipeline functions cell and the `run_matching_pipeline(...)`
   call.** Expect long runtimes for full multi-trial runs.
8. **Find outputs** under `OUTPUT_DIR/<timestamp>/`:
   - `idmatcher_identity_correspondences.csv` — the main result
   - `run_report.json` — per-trial diagnostics
   - `logs/` — one `.log` per trial with idmatcherai's stdout/stderr

### Troubleshooting

| Symptom | What to do |
|---|---|
| `FileNotFoundError: PROJECT_DIR not found` | Edit `PROJECT_DIR` in the Configuration cell to point at the actual folder name in your Drive. |
| `FileNotFoundError: SESSIONS_ROOT exists but contains no session_* folders` | Double-check that you uploaded the session folders to the right place — they must sit directly inside `checked_sessions_choice_exp/`. |
| `FileNotFoundError: trials summary GOT.xlsx … missing` (with pharma on) | Either upload the Excel to `PROJECT_DIR/datasets/` or set `DROP_PHARMA_GROUPS = False`. |
| `KeyError: trials summary GOT.xlsx is missing required column(s)` | The Excel must have columns named `Trial` and `Group` (case-sensitive). Restore them or replace the file. |
| `CUDA available: False` after enabling T4 | Re-do *Runtime → Change runtime type → T4 GPU*, then *Runtime → Restart runtime* and re-run all setup cells. Without a GPU the pipeline still works but is much slower. |
| `Cannot find idmatcherai` | The `pip install` cell didn't finish. Re-run it. If it fails, run `%pip install -q --upgrade idtrackerai` and restart the runtime. |
| `id_image_size mismatch across sessions` | Hard fail from the original script — segments of the same trial were tracked with different image sizes. Re-track the affected segments in idtracker.ai with the same image size. |
| `data_policy='...' — must be 'idmatcher.ai'` | Re-track the video in idtracker.ai with *Settings → Data policy → idmatcher.ai*, then re-upload and re-run. |